# 日历老化

支持两种存储方案：

- `activated`：每月静置后做容量检查、恢复检查并重新满充。
- `unactivated`：1、2、3……目标年限分别独立仿真，中途不循环，只在期末检查保持率和恢复率。

核心仿真与后处理由 `src.workflows.calendar_aging` 执行；本 Notebook 只保留配置、运行、绘图和导出入口。

## 1. 用户配置

In [ ]:
# 只修改本单元即可切换电芯、温度、存储方式和仿真规模
CONFIG = {
    "cell": "MIC",  # MIC=CW363；CW500=MICCW500
    "aging_mode": "activated",  # activated / unactivated
    "run_mode": "smoke",  # smoke / study
    "temperature_c": 25,
    "diagnostic_rate_c": 0.25,
    "days_per_month": 30,
    "days_per_year": 365,
    "diagnostic_period_minutes": 0.5,
    "modes": {
        "smoke": {
            "activated_months": 1,
            # 无活化 smoke：两个独立短期存储算例
            "unactivated_target_days": [1, 2],
            "rest_period_hours": 6,
            "showprogress": False,
            "return_solutions": True,
        },
        "study": {
            # 活化：20年 × 12个月
            "activated_months": 20 * 12,
            # 无活化：自动生成1、2、3……20年共20个独立算例
            "max_storage_years": 20,
            "storage_interval_years": 1,
            "rest_period_hours": 24,
            "showprogress": True,
            "return_solutions": True,
        },
    },
    "charge_cutoff_v": 3.65,
    "discharge_cutoff_v": 2.5,
    "output_name": "日历老化",
}


## 2. 环境与导入

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

SEARCH_ROOT = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in (SEARCH_ROOT, *SEARCH_ROOT.parents):
    if (candidate / "src" / "easy_imports.py").exists() and (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break
    nested = candidate / "BatteryProject"
    if (nested / "src" / "easy_imports.py").exists() and (nested / "pyproject.toml").exists():
        PROJECT_ROOT = nested
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Cannot locate BatteryProject root from {SEARCH_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook import setup_notebook
from src.workflows.calendar_aging import (
    CalendarAgingSpec,
    run_calendar_aging_workflow,
)

ctx = setup_notebook(cell=CONFIG["cell"], project_root=PROJECT_ROOT)


## 3. 运行日历老化

In [ ]:
spec = CalendarAgingSpec.from_mapping(CONFIG)
result = run_calendar_aging_workflow(
    spec,
    project_root=ctx.project_root,
    workspace_root=ctx.workspace_root,
)
metrics = result["metrics"]
rest_profiles = result["rest_profiles"]
display(metrics)
print("Run folder:", result["context"].run_dir)


## 4. 保持率、恢复率与自放电率

- `retention_rate`：静置后第一次放电容量 / 初始参考容量。
- `recovery_rate`：重新充满后的第二次放电容量 / 初始参考容量。
- `reversible_self_discharge_rate`：第一次放电与恢复后容量之间的可恢复差额。
- `irreversible_capacity_loss_rate`：恢复后仍未回到初始参考容量的不可恢复差额。

In [ ]:
x_col = "storage_years" if metrics["storage_years"].max() >= 1 else "storage_days"
x_label = "Storage Time (years)" if x_col == "storage_years" else "Storage Time (days)"
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(metrics[x_col], metrics["retention_rate"] * 100, "s-", label="Retention")
axes[0].plot(metrics[x_col], metrics["recovery_rate"] * 100, "o-", label="Recovery")
axes[0].set(xlabel=x_label, ylabel="Capacity (%)", title="Calendar Aging Capacity")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(metrics[x_col], metrics["reversible_self_discharge_rate"] * 100, "o-")
axes[1].set(xlabel=x_label, ylabel="Self-discharge (%)", title="Reversible Self-discharge")
axes[1].grid(True)
fig.tight_layout()
plt.show()


## 5. 静置电压曲线叠加

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for label, frame in rest_profiles.groupby("case", sort=False):
    ax.plot(frame["rest_time_h"] / 24, frame["voltage_v"], label=label)
ax.set(xlabel="Rest Time (days)", ylabel="Voltage (V)", title="Voltage During Storage")
ax.grid(True)
if rest_profiles["case"].nunique() <= 20:
    ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
plt.show()


## 6. 期末充放电曲线叠加（无活化）

无活化模式下，每个目标年限都是独立 Solution。以下单元叠加各算例期末的第一次放电（存储后保持容量）和第二次放电（充电恢复后容量）。

In [ ]:
if spec.aging_mode != "unactivated":
    print("当前为 activated 模式；该图仅用于 unactivated。")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
    for label, solution in result["solutions"].items():
        endpoint = solution.cycles[1]
        discharge_steps = []
        for step in endpoint.steps:
            current = step["Current [A]"].entries.reshape(-1)
            if current.max() > 1e-6:
                discharge_steps.append(step)
        for axis, step, title in zip(
            axes,
            discharge_steps,
            ("After Storage", "After Recharge"),
        ):
            capacity = step["Discharge capacity [A.h]"].entries.reshape(-1)
            voltage = step["Voltage [V]"].entries.reshape(-1)
            axis.plot(capacity - capacity[0], voltage, label=label)
            axis.set(xlabel="Discharge Capacity (Ah)", title=title)
            axis.grid(True)
    axes[0].set_ylabel("Voltage (V)")
    axes[1].legend(ncol=2, fontsize=8)
    fig.tight_layout()
    plt.show()


## 7. 输出文件

In [ ]:
print("Metrics:", result["metrics_path"])
print("Rest voltage profiles:", result["rest_profiles_path"])
# CSV 已由 workflow 写入标准 run folder；无需在 examples 目录落地中间结果。
